In [ ]:
from lightning.pytorch.utilities import CombinedLoader

from sled_geo.model import SLEDTrainingFramework, ModalityDetails
from sled_geo import get_encoder as position_encoder_access
from torchgeo.datasets import S2100k # need to download from Huggingface
from torchgeo.models import ResNet18_Weights
import timm

import os
from torch.utils.data import DataLoader, random_split
from lightning.pytorch.loggers import TensorBoardLogger
from lightning import  seed_everything, Trainer, LightningDataModule
import torch

from lightning.pytorch.callbacks import EarlyStopping

In [ ]:
class SampleDataModule(LightningDataModule):
    def __init__(
            self,
            batch_size: int = 64,
            num_workers: int = 6,
            val_random_split_fraction: float = 0.1,
            data_dir: str = "data",
    ):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.batch_size = batch_size
        self.num_workers = num_workers

        self.val_random_split_fraction = val_random_split_fraction
        self.save_hyperparameters()

    def prepare_data(self) -> None:
        if not os.path.exists(self.data_dir):
            print("""
            No dataset found. To download, please follow instructions on: https://github.com/microsoft/satclip
            """)

    def setup(self, stage="fit"):
        dataset = S2100k(root = self.data_dir)

        N_val = int(len(dataset) * self.val_random_split_fraction)
        N_train = len(dataset) - N_val
        self.train_dataset, self.val_dataset = random_split(dataset, [N_train, N_val])

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            shuffle=False,
            # persistent_workers=True if self.num_workers > 0 else False,
        )

    def test_dataloader(self):
        raise NotImplementedError

In [ ]:
# setup
seed_everything(42)
torch.set_float32_matmul_precision('high')
tb_logger = TensorBoardLogger("your_run_name_here")

# Grab whatever position encoder you'd like to use; here, we're using RFF, as popularized by GeoCLIP
num_dim_for_position_embedding = 768
position_encoder = position_encoder_access.get_rff_encoder(embed_dim=num_dim_for_position_embedding)

# Grab an encoder for Sentinel-2 data . . . let's use a quick MOCO pre-trained model
weights = ResNet18_Weights.SENTINEL2_ALL_MOCO
s2_encoder = timm.create_model("resnet18", in_chans=weights.meta["in_chans"], num_classes=512)
s2_encoder.load_state_dict(weights.get_state_dict(), strict=False)

# spin up modality details for Sentinel-2; we do this to enable flexible use with existing models and
# datasets, many of which have slightly different naming conventions
sentinel_2_key = "s2"
sentinel_2_details = ModalityDetails(
    modality_key=sentinel_2_key,
    encoded_dimensions=512,
    encoder = s2_encoder,
    coordinate_key="point", # from the S2-100k dataset
    data_key="image" # also from the S2-100k dataset
)

modality_details = [sentinel_2_details] # do this with as many modes as you'd like!

In [ ]:
# create datamodule
batch_size = 128  # we've gotten results with as low as 32
sentinel_2_datamodule = SampleDataModule(data_dir="data", # points to wherever you've downloaded S2-100K
                                         batch_size=batch_size,
                                         num_workers=12, # check how many CPUs you have available!
                                         val_random_split_fraction=0.1 # set to 0 if you don't want early stopping
                                         )

# specify combined dataloaders (needed even if you're just using one modality)
# Your key for the dataloader must be the same as the key for the modality details!
train_dataloaders = CombinedLoader({sentinel_2_key, sentinel_2_datamodule.train_dataloader()},
                                  mode="max_size")
val_dataloaders = CombinedLoader({sentinel_2_key, sentinel_2_datamodule.val_dataloader()},
                                mode="max_size")

In [ ]:
# create our SLED framework
your_sled_framework = SLEDTrainingFramework(modality_details,
                                            position_encoder=position_encoder,
                                            lr=0.001,
                                            output_dim=num_dim_for_position_embedding,
                                            batch_size=batch_size, # parameter here to avoid Pytorch Lightning warning bug
                                            use_alignment_heads=False, # Set to true when training against multiple modes
                                            )

# Add an early stopping callback
early_stopping = EarlyStopping(
    monitor="val_early_stopping_loss",  # Track validation loss
    min_delta=0.001,  # Minimum change to qualify as improvement
    patience=20,  # Stop after 10 epochs with no improvement
    verbose=True,  # Print when stopping
    mode="min"  # "min" for loss, "max" for accuracy
)

# Can add lots of additional callbacks such as checkpointing, early stopping, or progress bars!
# Optional: if you don't want early stopping, simply remove the callback=[early_stopping] line
trainer = Trainer(limit_train_batches=1.0, max_epochs=100, logger=tb_logger, callbacks=[early_stopping])

trainer.fit(your_sled_framework, train_dataloaders=train_dataloaders, val_dataloaders=val_dataloaders)

trained_position_encoder = your_sled_framework.position_encoder # get your trained position encoder!